### Task1 LOAD DATA

In [1]:
import pandas as pd
import re

### Task2 read all CSV files

In [2]:
categories = pd.read_csv("categories.csv")
employees = pd.read_csv("employees.csv")
prescriptions = pd.read_csv("prescriptions_log.csv")
products = pd.read_csv("products_catalog.csv")

sales1 = pd.read_csv("sales_group1_storesA.csv")
sales2 = pd.read_csv("sales_group2_storesB.csv")
sales3 = pd.read_csv("sales_group3_storesC.csv")

stores = pd.read_csv("stores.csv")
suppliers = pd.read_csv("suppliers.csv")
stock = pd.read_csv("stock_condition_log.csv")
customers = pd.read_csv("customers_master.csv")

In [3]:
print(categories.head())
print(products.head())
print(sales1.head())
print(customers.head())

   category_id           category_name
0            1      Prescription Drugs
1            2         OTC Pain Relief
2            3  Vitamins & Supplements
3            4           Personal Care
4            5         Medical Devices
   product_id      product_name  category_id  supplier_id  unit_price  \
0           1  Health_Product_1            4            2       98.97   
1           2  Health_Product_2            8            3      111.42   
2           3  Health_Product_3            9            4       14.86   
3           4  Health_Product_4            5            4      274.78   
4           5  Health_Product_5            2            3       87.09   

   cost_price      brand expiry_date storage_condition  created_at  \
0      107.40     BioZen  2026-11-29      Refrigerated  2022-03-12   
1       64.75   MediCare  2026-08-08      Refrigerated  2022-09-13   
2        7.49    CuraMax  2027-03-15      Refrigerated  2023-06-25   
3      153.13  VitalMeds  2029-09-28        Con

In [4]:
print(sales1.columns)
print(sales2.columns)
print(sales3.columns)

Index(['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id',
       'quantity', 'sale_date', 'unit_price', 'discount_pct',
       'payment_method'],
      dtype='str')
Index(['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id',
       'qty', 'sale_date', 'amount', 'discount_pct', 'payment_method'],
      dtype='str')
Index(['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id',
       'quantity', 'sale_date', 'unit_price', 'discount_pct', 'payment_method',
       'notes'],
      dtype='str')


In [5]:
sales2 = sales2.rename(columns={
    "qty": "quantity",
    "amount": "sales_amount"
})

In [6]:
sales1 = sales1.rename(columns={
    "unit_price": "sales_amount"
})

### Task 3 Cleaning Dates

In [7]:
products["expiry_date"] = pd.to_datetime(
    products["expiry_date"],
    errors="coerce"
)

In [8]:
employees["hire_date"] = pd.to_datetime(
    employees["hire_date"],
    errors="coerce"
)

In [9]:
sales1["sale_date"] = pd.to_datetime(
    sales1["sale_date"],
    dayfirst=True,
    errors="coerce"
)

sales2["sale_date"] = pd.to_datetime(
    sales2["sale_date"],
    dayfirst=True,
    errors="coerce"
)

sales3["sale_date"] = pd.to_datetime(
    sales3["sale_date"],
    dayfirst=True,
    errors="coerce"
)

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_13004\2279965415.py:7: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  sales2["sale_date"] = pd.to_datetime(


In [10]:
sales1["sale_date"] = sales1["sale_date"].dt.strftime("%d/%m/%Y")

### Task 4 Cleaning Prices

In [11]:
sales2["sales_amount"] = (
    sales2["sales_amount"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.strip()
)

In [12]:
sales2["sales_amount"] = pd.to_numeric(
    sales2["sales_amount"],
    errors="coerce"
)

In [13]:
products["unit_price"] = pd.to_numeric(
    products["unit_price"],
    errors="coerce"
)

products["cost_price"] = pd.to_numeric(
    products["cost_price"],
    errors="coerce"
)

### Task 5 Cleaning Doctor Names

In [14]:
prescriptions["doctor_name"] = (
    prescriptions["doctor_name"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [15]:
prescriptions["doctor_name"] = (
    prescriptions["doctor_name"]
    .str.replace(r"^Dr\.?\s*", "Dr. ", regex=True)
)

### Task 6 Cleaning Dosage

In [16]:
prescriptions["dosage"] = (
    prescriptions["dosage"]
    .astype(str)
    .str.strip()
)

In [17]:
prescriptions["dosage"] = prescriptions["dosage"].str.replace(
    r"\bhrs?\b",
    "hours",
    regex=True,
    case=False
)

In [18]:
prescriptions["dosage"] = prescriptions["dosage"].str.replace(
    r"(\d+)\s*ml",
    r"\1 ml",
    regex=True,
    case=False
)

### Task 7 Extracting pack_info Data

In [19]:
products["pack_size"] = products["pack_info"].str.extract(
    r"Pack:\s*([^|/]+)",
    expand=False
)

In [20]:
products["strength"] = products["pack_info"].str.extract(
    r"Strength:\s*([^|/]+)",
    expand=False
)

In [21]:
products["form"] = products["pack_info"].str.extract(
    r"Form:\s*([^|/]+)",
    expand=False
)

In [22]:
products["pack_size"] = products["pack_size"].str.strip()
products["strength"] = products["strength"].str.strip()
products["form"] = products["form"].str.strip()

### Task 8 Extracting Promo Codes

In [23]:
sales3["promo_code"] = sales3["notes"].str.extract(
    r"Promo code\s+([A-Za-z0-9]+)",
    expand=False
)

### Task 9 Merging Sales Files

In [24]:
sales_all = pd.concat(
    [sales1, sales2, sales3],
    ignore_index=True
)

In [25]:
print(sales_all.shape)
print(sales_all.columns)

(9950, 13)
Index(['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id',
       'quantity', 'sale_date', 'sales_amount', 'discount_pct',
       'payment_method', 'unit_price', 'notes', 'promo_code'],
      dtype='str')


### Task 10 Joining Sales with Products

In [26]:
sales_all = sales_all.merge(
    products[
        [
            "product_id",
            "product_name",
            "category_id",
            "supplier_id",
            "cost_price"
        ]
    ],
    on="product_id",
    how="left"
)

### Task 11 Calculating Cost

In [27]:
sales_all["cost_amount"] = (
    sales_all["quantity"] *
    sales_all["cost_price"]
)

### Task 12 Calculating Profit

In [28]:
sales_all["profit"] = (
    sales_all["sales_amount"] -
    sales_all["cost_amount"]
)

In [29]:
sales_all["profit"] = (
    sales_all["sales_amount"] -
    sales_all["cost_amount"]
)

### Task 13 Cleaning Customers

In [30]:
customers["full_name"] = (
    customers["full_name"]
    .str.strip()
)

In [31]:
customers["city"] = (
    customers["city"]
    .str.strip()
    .str.title()
)

In [32]:
customers["phone"] = (
    customers["phone"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

In [33]:
customers["phone"] = (
    customers["phone"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

In [34]:
customers = pd.read_csv(
    "customers_master.csv",
    dtype={"phone": "string"}
)

### Task 14 Extracting Loyalty Status

In [35]:
customers["loyalty_status"] = customers["notes"].str.extract(
    r"^(\w+)\s+member",
    expand=False
)

In [36]:
customers["loyalty_since"] = customers["notes"].str.extract(
    r"loyalty since\s+(\d{4})",
    expand=False
)

In [37]:
customers["loyalty_since"] = customers["notes"].str.extract(
    r"loyalty since\s+(\d{4})",
    expand=False
)

### Task 15 Detecting Duplicates

In [38]:
duplicates = customers[
    customers.duplicated(
        subset=["email"],
        keep=False
    )
]

print(duplicates)

Empty DataFrame
Columns: [customer_id, full_name, email, phone, city, signup_date, notes, loyalty_status, loyalty_since]
Index: []


In [39]:
customers = customers.drop_duplicates(
    subset=["email"],
    keep="first"
)

### Task 16 Merging Customers with Sales

In [40]:
sales_all = sales_all.merge(
    customers[
        [
            "customer_id",
            "full_name",
            "email",
            "phone",
            "city",
            "loyalty_status",
            "loyalty_since"
        ]
    ],
    on="customer_id",
    how="left"
)

### Task 17 — Merge Employees

In [41]:
sales_all = sales_all.merge(
    employees[
        [
            "employee_id",
            "employee_name",
            "position",
            "is_pharmacist"
        ]
    ],
    on="employee_id",
    how="left"
)

### Task 18 — Merge Stores

In [42]:
sales_all = sales_all.merge(
    stores[
        [
            "store_id",
            "store_name",
            "region",
            "city",
            "manager_name"
        ]
    ],
    on="store_id",
    how="left"
)

### Task 19 Data Quality Check

In [43]:
print(sales_all.isna().sum())

sale_id              0
store_id             0
employee_id          0
product_id           0
customer_id          0
quantity             0
sale_date         1996
sales_amount      3373
discount_pct         0
payment_method       0
unit_price        6577
notes             8294
promo_code        9095
product_name         0
category_id          0
supplier_id          0
cost_price           0
cost_amount          0
profit            3373
full_name            0
email                0
phone                0
city_x               0
loyalty_status    7005
loyalty_since     7005
employee_name        0
position             0
is_pharmacist        0
store_name           0
region               0
city_y               0
manager_name         0
dtype: int64


In [44]:
print(sales_all.duplicated().sum())

50


In [45]:
print(sales_all.dtypes)

sale_id             int64
store_id            int64
employee_id         int64
product_id          int64
customer_id         int64
quantity            int64
sale_date          object
sales_amount      float64
discount_pct      float64
payment_method        str
unit_price        float64
notes                 str
promo_code            str
product_name          str
category_id         int64
supplier_id         int64
cost_price        float64
cost_amount       float64
profit            float64
full_name             str
email                 str
phone              string
city_x                str
loyalty_status        str
loyalty_since         str
employee_name         str
position              str
is_pharmacist       int64
store_name            str
region                str
city_y                str
manager_name          str
dtype: object


In [46]:
print(sales_all.shape)

(9950, 32)


### Task 20 Checking IDs

In [47]:
missing_products = sales_all[
    sales_all["cost_price"].isna()
]

print(missing_products[["product_id"]])

Empty DataFrame
Columns: [product_id]
Index: []


In [48]:
missing_customers = sales_all[
    sales_all["full_name"].isna()
]

print(missing_customers[["customer_id"]])

Empty DataFrame
Columns: [customer_id]
Index: []


### Task 21 Saving the Final Dataset

In [49]:
sales_all.to_csv(
    "cleaned_sales.csv",
    index=False
)

In [50]:
customers.to_csv(
    "cleaned_customers.csv",
    index=False
)